In [23]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path to dataset files: /kaggle/input/cardekho-used-car-data


In [24]:
import os

# Check what files are inside the folder
print(os.listdir(path))

['cardekho_dataset.csv']


In [25]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(path, "cardekho_dataset.csv"))

print(df.shape)
print(df.columns)
df.head()

(15411, 14)
Index(['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven',
       'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine',
       'max_power', 'seats', 'selling_price'],
      dtype='object')


,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


TASK-1: CLEANING

In [26]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [27]:
df["mileage"] = pd.to_numeric(df["mileage"], errors="coerce")
df["engine"] = pd.to_numeric(df["engine"], errors="coerce")
df["max_power"] = pd.to_numeric(df["max_power"], errors="coerce")

for col in ["mileage", "engine", "max_power"]:
    df[col].fillna(df[col].median(), inplace=True)

df = df[(df["selling_price"] >= 10000) & (df["selling_price"] != 999999999)]

df = df.drop_duplicates()

print(df.shape)
df.head()

(15244, 13)


/tmp/ipykernel_1815/3311007367.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


TASK 2 (Encoding)

In [28]:
df['transmission_type'] = df['transmission_type'].map({
    'Manual': 0,
    'Automatic': 1
})

In [29]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

In [30]:
print(df.columns.tolist())

['car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


TASK 3 — Split data + Baseline MAE

In [31]:
X = df.drop(columns=["selling_price"])
y = df["selling_price"]

In [32]:
X = X.select_dtypes(include=['int64', 'float64', 'uint8'])

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [34]:
import numpy as np
from sklearn.metrics import mean_absolute_error

baseline_pred = np.full_like(y_test, y_train.mean())

mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline MAE: ₹", round(mae))

Baseline MAE: ₹ 440053
